## Arcface loss

In [ ]:
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '' # _detected', '_detected_manual'

In [ ]:
import torch
from proportional_split_xy import proportional_split_xy
# Load the saved embeddings
embeddings = f'saved_models/{megadescriptor_version}/embeddings/emb{detection}.pt'
labels = f'saved_models/{megadescriptor_version}/labels/labels{detection}.pt'
label_encoder = f'saved_models/{megadescriptor_version}/label_encoders/label_encoder{detection}.pkl'

all_embeddings = torch.load(embeddings)

print(all_embeddings.shape)  # torch.Size([260, 768])

In [ ]:
import torch, joblib
label_ids = torch.load(labels, weights_only=False)
encoder = joblib.load(label_encoder)

# Convert names back later:
names = encoder.inverse_transform(label_ids)


In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Example setup
X = all_embeddings.float()
y = torch.from_numpy(label_ids).long()   # shape (260,)

dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)


In [ ]:
if __name__ == "__main__":
    import os, random, numpy as np, torch, torch.nn.functional as F
    from sklearn.model_selection import train_test_split  # (can remove)
    import torch.nn as nn
    import torch.optim as optim
    import pandas as pd

    # ---------- reproducibility ----------
    seed = 0
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # ---------- load embeddings and labels ----------

    embeddings = torch.load(embeddings)
    if isinstance(embeddings, np.ndarray):
        embeddings = torch.from_numpy(embeddings)
    embeddings = embeddings.float()
    N, D = embeddings.shape
    print("Loaded embeddings:", embeddings.shape)

    try:
        labels_obj = torch.load(labels, weights_only=False)
    except TypeError:
        labels_obj = torch.load(labels)
    except Exception as e:
        try:
            labels_obj = np.load(labels, allow_pickle=True)
        except Exception:
            raise RuntimeError(f"Unable to load labels file {labels}: {e}")

    if isinstance(labels_obj, torch.Tensor):
        labels = labels_obj.long()
    else:
        labels = torch.from_numpy(np.array(labels_obj)).long()
    assert labels.shape[0] == N, f"Labels length {labels.shape[0]} != embeddings {N}"
    print("Loaded labels:", labels.shape, "num_classes:", len(torch.unique(labels)))

    # ---------- proportional split ----------
    X_train, X_val, y_train, y_val = proportional_split_xy(embeddings.numpy(), labels.numpy(), query_ratio=0.2)
    print("Train/Val sizes:", len(X_train), len(X_val))

    # Convert to torch tensors
    X_train = torch.from_numpy(np.array(X_train)).float().to(device)
    y_train = torch.from_numpy(np.array(y_train)).long().to(device)
    X_val = torch.from_numpy(np.array(X_val)).float().to(device)
    y_val = torch.from_numpy(np.array(y_val)).long().to(device)

    # ---------- ArcFace head implementation ----------
    class ArcFaceHead(nn.Module):
        def __init__(self, in_features, out_features, s=30.0, m=0.5):
            super().__init__()
            self.s = s
            self.m = m
            self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
            nn.init.xavier_uniform_(self.weight)

        def forward(self, x, labels=None):
            x_norm = F.normalize(x, p=2, dim=1)
            W = F.normalize(self.weight, p=2, dim=1)
            cosine = torch.matmul(x_norm, W.t()).clamp(-1.0, 1.0)
            if labels is None:
                return cosine * self.s
            theta = torch.acos(cosine)
            target_logits = torch.cos(theta + self.m)
            one_hot = torch.zeros_like(cosine)
            one_hot.scatter_(1, labels.view(-1, 1), 1.0)
            logits = cosine * (1 - one_hot) + target_logits * one_hot
            logits = logits * self.s
            loss = F.cross_entropy(logits, labels)
            return loss, logits

    class HeadModel(nn.Module):
        def __init__(self, input_dim, feat_dim=256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, feat_dim),
                nn.BatchNorm1d(feat_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(feat_dim, feat_dim // 2),
                nn.BatchNorm1d(feat_dim // 2),
                nn.ReLU(inplace=True),
            )
        def forward(self, x):
            return self.net(x)

    num_classes = int(len(torch.unique(labels)))
    feat_dim = 256
    head = HeadModel(D, feat_dim=feat_dim).to(device)
    arc = ArcFaceHead(in_features=feat_dim//2, out_features=num_classes, s=30.0, m=0.5).to(device)

    optimizer = optim.AdamW(list(head.parameters()) + list(arc.parameters()), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)

    # ---------- training ----------
    num_epochs = 100
    batch_size = 16
    best_val_acc = 0.0
    best_state = None

    for epoch in range(num_epochs + 1):
        head.train()
        arc.train()
        if epoch != 0:
            
            perm = torch.randperm(X_train.size(0), device=device)
            total_loss = 0.0
            total_samples = 0

            for i in range(0, len(perm), batch_size):
                batch_ids = perm[i:i+batch_size]
                batch_emb = X_train[batch_ids]
                batch_labels = y_train[batch_ids]

                optimizer.zero_grad()
                features = head(batch_emb)
                loss, _ = arc(features, batch_labels)
                loss.backward()
                optimizer.step()

                total_loss += float(loss.item()) * batch_emb.size(0)
                total_samples += batch_emb.size(0)

            avg_loss = total_loss / total_samples

            # ---------- validation ----------
            head.eval()
            arc.eval()
        with torch.no_grad():
            val_emb = head(X_val)
            logits = torch.matmul(F.normalize(val_emb, p=2, dim=1),
                                  F.normalize(arc.weight, p=2, dim=1).t()) * arc.s
            preds = logits.argmax(dim=1)
            val_acc = (preds == y_val).float().mean().item()

        print(f"Epoch {epoch:02d} | Train loss: {avg_loss:.4f} | Val acc: {val_acc:.4f}")
        if epoch != 0:
            scheduler.step(avg_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {
                "head": head.state_dict(),
                "arc": arc.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
                "val_acc": val_acc,
            }
            torch.save(best_state, "best_arcface_state.pt")

    print("\n✅ Best validation accuracy:", best_val_acc)

    if best_state is not None:
        head.load_state_dict(best_state["head"])
        arc.load_state_dict(best_state["arc"])

    head.eval()
    with torch.no_grad():
        calibrated_feats = head(embeddings.to(device))
        calibrated_feats = F.normalize(calibrated_feats, p=2, dim=1).cpu()
        torch.save(calibrated_feats, "emb_arcface_calibrated.pt")
        print("Saved calibrated embeddings: emb_arcface_calibrated.pt")

    torch.save({"head": head.state_dict(), "arc": arc.state_dict()}, "arcface_model_final.pt")
    print("Saved model state: arcface_model_final.pt")
